# Fine-tune a small language model on the Menstrual Health QnA dataset (QLoRA)

**Kaggle setup before running**
1. Add the dataset `atkiyamaisha/menst-and-menstrual-health-awareness-merged` as an input (the code finds the CSV wherever Kaggle mounts it).
2. Settings: Accelerator = **GPU T4 x2** (recommended; the notebook uses only the first GPU, on purpose) and Internet = **On** (to download the model).
3. Run all cells top to bottom. Do not run cells out of order, and use *Restart & Run All* if you change the config.

**Model.** `Qwen/Qwen2.5-0.5B-Instruct`: ungated and tiny. It trains about 3x faster than the 1.5B model and, quantized to 4-bit GGUF (section 9), is small enough for a 512 MB free host. Answers will be less accurate than the 1.5B model.
Section 9 converts the merged model to GGUF (Q4_K_M and Q3_K_M) for llama.cpp.

**Safety note.** This is a health education model. It can still produce wrong statements.
Always show a disclaimer and advise users to see a doctor for symptoms.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets rouge_score
# Kaggle ships torchao 0.10; new peft raises ImportError on it when it wraps a normal fp16 model (merge step).
!pip uninstall -y -q torchao

In [ ]:
import os
# --- Kaggle crash guards (must run BEFORE torch is imported) ---
# 1) Use ONE GPU. On "T4 x2" the HF Trainer would wrap the 4-bit model in DataParallel, which crashes with bitsandbytes.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

import glob, re, math, random, shutil, gc, inspect, zipfile
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorForSeq2Seq,
                          EarlyStoppingCallback, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In Kaggle open Settings -> Accelerator -> GPU T4 x2, then run again.")

# ---------------- config ----------------
MODEL_ID   = "Qwen/Qwen2.5-0.5B-Instruct"   # small enough for a 512 MB free host once quantized
DATA_DIR   = "/kaggle/input/menst-and-menstrual-health-awareness-merged"
OUT_DIR    = "/kaggle/working/menst-slm-lora"      # final adapter (small)
CKPT_DIR   = "/kaggle/working/ckpt"                # training checkpoints (deleted after saving the adapter)
MAX_LEN    = 512        # prompt + answer tokens
SEED       = 42
EPOCHS     = 3
LR         = 2e-4
BATCH      = 4          # per-device batch size (small: Qwen's 152k-token vocabulary makes logits large)
GRAD_ACCUM = 4          # effective batch = BATCH x GRAD_ACCUM = 16
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EVAL_BATCH = 8          # batch size for test-set generation
QUICK_TEST = False      # True = tiny run to check that everything works

SYSTEM_PROMPT = (
    "You are a menstrual health education assistant. Give clear, simple, supportive, "
    "general information. For severe pain, very heavy bleeding, missed periods for three "
    "months or more, possible pregnancy, or any worrying symptom, advise seeing a doctor. "
    "If you are not sure, say so."
)

set_seed(SEED)
BF16 = torch.cuda.get_device_capability()[0] >= 8   # T4/P100 -> False -> fp16
DTYPE = torch.bfloat16 if BF16 else torch.float16
DEVICE_MAP = {"": 0}
print("GPU:", torch.cuda.get_device_name(0), "| visible GPUs:", torch.cuda.device_count(), "| bf16:", BF16)
if torch.cuda.device_count() > 1:
    print("WARNING: more than one GPU is visible (torch was imported before the guard ran). "
          "Use Run -> Restart & Run All so only GPU 0 is used.")

## 1. Load data and split by answer (no answer leaks between train, validation and test)

In [ ]:
csvs = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)) or \
       sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
print("CSV files found:", csvs)
if not csvs:
    raise FileNotFoundError("No CSV under /kaggle/input. Add the dataset with 'Add Input' in the right sidebar.")

def read_csv_any(path):
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {path}")

def pick_columns(d):
    low = {str(c).strip().lower(): c for c in d.columns}
    q = next((low[k] for k in low if "question" in k), None)
    a = next((low[k] for k in low if "answer" in k), None)
    return q, a

df = None
for path in csvs:                                   # use the first CSV that has a question and an answer column
    raw = read_csv_any(path)
    qc, ac = pick_columns(raw)
    if qc is not None and ac is not None and qc != ac:
        df = raw[[qc, ac]].rename(columns={qc: "Question", ac: "Answer"})
        print("Using:", path, "| columns:", qc, "/", ac)
        break
if df is None:
    raise ValueError("No CSV with Question and Answer columns. Columns seen: " +
                     str([list(read_csv_any(p).columns) for p in csvs[:3]]))

df = df.dropna()
df["Question"] = df["Question"].astype(str).str.strip()
df["Answer"]   = df["Answer"].astype(str).str.strip()
df = df[(df["Question"] != "") & (df["Answer"] != "")].drop_duplicates().reset_index(drop=True)
print("Rows:", len(df), "| unique answers:", df["Answer"].nunique())

# The dataset has many paraphrased questions per answer.
# Splitting by answer keeps near-duplicates out of validation and test.
groups = df["Answer"].map(lambda s: re.sub(r"\s+", " ", s).lower())
if groups.nunique() < 10:
    raise ValueError(f"Only {groups.nunique()} unique answers; too few to split into train/val/test.")
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, hold_idx = next(gss1.split(df, groups=groups))
hold = df.iloc[hold_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_i, test_i = next(gss2.split(hold, groups=groups.iloc[hold_idx]))
train_df, val_df, test_df = df.iloc[train_idx], hold.iloc[val_i], hold.iloc[test_i]

for a, b in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert not (set(groups[a.index]) & set(groups[b.index])), "answer leak between splits"
print("train/val/test:", len(train_df), len(val_df), len(test_df))

if QUICK_TEST:
    train_df, val_df, test_df = train_df.head(256), val_df.head(64), test_df.head(32)

## 2. Tokenise (loss is computed on the answer only)

In [ ]:
try:
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
except Exception as e:
    raise RuntimeError("Could not download the tokenizer. In Kaggle turn Internet = On (Settings), "
                       "or check MODEL_ID / your HF token for gated models.") from e
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def chat_prompt(question):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def encode(ex):
    p_ids = tok(chat_prompt(ex["Question"]), add_special_tokens=False)["input_ids"]
    a_ids = tok(ex["Answer"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

def to_ds(d):
    ds = Dataset.from_pandas(d.reset_index(drop=True), preserve_index=False)
    ds = ds.map(encode, remove_columns=["Question", "Answer"])
    # drop rows whose prompt alone fills MAX_LEN (no answer tokens -> NaN loss)
    return ds.filter(lambda ex: any(l != -100 for l in ex["labels"]))

train_ds, val_ds = to_ds(train_df), to_ds(val_df)
print("kept after filtering: train", len(train_ds), "| val", len(val_ds))
lens = [len(x) for x in train_ds["input_ids"]]
print("token length: mean %.0f, 95th pct %.0f, max %d (MAX_LEN=%d)" %
      (np.mean(lens), np.percentile(lens, 95), max(lens), MAX_LEN))

## 3. Load the model in 4-bit and attach LoRA adapters

In [ ]:
def load_base(**kw):
    """Load the base model; works with old (torch_dtype) and new (dtype) transformers."""
    try:
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, **kw)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, **kw)

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE)

model = load_base(quantization_config=bnb, device_map={"": 0})
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False})

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)

# fp16 training needs fp32 trainable weights, otherwise: "Attempting to unscale FP16 gradients"
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
model.print_trainable_parameters()

## 4. Train

In [ ]:
n_epochs = 1 if QUICK_TEST else EPOCHS
total_steps = math.ceil(len(train_ds) / (BATCH * GRAD_ACCUM)) * n_epochs
WARMUP_STEPS = max(1, int(0.03 * total_steps))   # explicit steps work in old and new transformers

# the argument was renamed evaluation_strategy -> eval_strategy; support both
_params = inspect.signature(TrainingArguments.__init__).parameters
EVAL_KEY = "eval_strategy" if "eval_strategy" in _params else "evaluation_strategy"

args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=n_epochs,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    fp16=not BF16, bf16=BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=25,
    remove_unused_columns=False,      # the dataset already holds only input_ids / attention_mask / labels
    dataloader_num_workers=0,
    report_to="none",
    seed=SEED,
    **{EVAL_KEY: "epoch"},
)
if getattr(args, "n_gpu", 1) > 1:     # belt and braces: never use DataParallel with a 4-bit model
    try:
        args._n_gpu = 1
    except Exception:
        pass

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()
m = trainer.evaluate()
print("validation loss: %.4f | perplexity: %.2f" % (m["eval_loss"], math.exp(m["eval_loss"])))

## 5. Save the adapter and the tokenizer

In [ ]:
trainer.save_model(OUT_DIR)
tok.save_pretrained(OUT_DIR)
shutil.rmtree(CKPT_DIR, ignore_errors=True)     # checkpoints are big and no longer needed
print(os.listdir(OUT_DIR))

## 6. Chat with the model and score it on the held-out test set

In [ ]:
model.config.use_cache = True
model.eval()

@torch.no_grad()
def ask_batch(questions, max_new_tokens=256):
    prompts = [chat_prompt(q) for q in questions]
    old_side = tok.padding_side
    tok.padding_side = "left"                      # decoder-only models must be left-padded for generation
    try:
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    finally:
        tok.padding_side = old_side
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, top_k=None,
                         repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    plen = enc["input_ids"].shape[1]
    return [tok.decode(o[plen:], skip_special_tokens=True).strip() for o in out]

def ask(question, max_new_tokens=256):
    return ask_batch([question], max_new_tokens)[0]

for q in ["Is it normal to miss my period for three months?",
          "How can I reduce period cramps?"]:
    print("Q:", q, "\nA:", ask(q), "\n")

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

N = min(len(test_df), 32 if QUICK_TEST else 200)
sample = test_df.sample(N, random_state=SEED)
qs, refs = sample["Question"].tolist(), sample["Answer"].tolist()
preds = []
for i in range(0, N, EVAL_BATCH):
    preds += ask_batch(qs[i:i + EVAL_BATCH])
    torch.cuda.empty_cache()

scores = [(scorer.score(r, p)["rouge1"].fmeasure, scorer.score(r, p)["rougeL"].fmeasure)
          for r, p in zip(refs, preds)]
r1, rl = np.mean(scores, axis=0)
print(f"Test samples: {N} | ROUGE-1 F1: {r1:.3f} | ROUGE-L F1: {rl:.3f}")
res = pd.DataFrame({"Question": qs, "Reference": refs, "Prediction": preds})
res.to_csv("/kaggle/working/test_predictions.csv", index=False)
res.head()

## 7. Download the adapter (small file)

The adapter folder is small. To use it later:
```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, "menst-slm-lora")
```

In [ ]:
from IPython.display import FileLink, display
os.chdir("/kaggle/working")
shutil.make_archive("menst-slm-lora", "zip", OUT_DIR)
display(FileLink("menst-slm-lora.zip"))
display(FileLink("test_predictions.csv"))

## 8. Standalone model for building the chatbot (recommended)

This merges the adapter into the base model, so the downloaded folder loads with plain `transformers`.
It reloads the base in fp16 because merging into the 4-bit model would lose precision.
The folder also gets a ready-made `chat_app.py` (Gradio chat window) and a short README.

In [ ]:
from peft import PeftModel

# free GPU memory from training (safe to re-run)
for _name in ("trainer", "model", "base", "merged", "enc"):
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache()

# Guard: if an old torchao is still installed, tell peft to ignore it (the install cell already uninstalls it)
import importlib
for _mod in ("peft.import_utils", "peft.tuners.lora.torchao", "peft.utils.quantization_utils"):
    try:
        importlib.import_module(_mod).is_torchao_available = lambda: False
    except Exception:
        pass

MERGED_DIR = "/kaggle/working/menst-slm-merged"
base = load_base(device_map=DEVICE_MAP)
merged = PeftModel.from_pretrained(base, OUT_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tok.save_pretrained(MERGED_DIR)

APP = '''# Run:  pip install transformers accelerate torch gradio
#       python chat_app.py            (loads the model from this folder)
import sys, torch, gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

PATH = sys.argv[1] if len(sys.argv) > 1 else "."
SYSTEM_PROMPT = __SYSTEM__
gpu = torch.cuda.is_available()
tok = AutoTokenizer.from_pretrained(PATH)
model = AutoModelForCausalLM.from_pretrained(
    PATH, torch_dtype=torch.float16 if gpu else torch.float32,
    device_map="auto" if gpu else None)
model.eval()

@torch.no_grad()
def reply(message, history):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": message}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=256, do_sample=False,
                         repetition_penalty=1.1, pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

gr.ChatInterface(
    reply, title="Menstrual Health Assistant",
    description="General education only. This is not medical advice. See a doctor for worrying symptoms.",
).launch()
'''
with open(os.path.join(MERGED_DIR, "chat_app.py"), "w") as f:
    f.write(APP.replace("__SYSTEM__", repr(SYSTEM_PROMPT)))

with open(os.path.join(MERGED_DIR, "README.txt"), "w") as f:
    f.write(f"Luna Loves You: fine-tuned from {MODEL_ID} on the Menstrual Health QnA dataset (QLoRA, merged to fp16).\n"
            "Load with AutoModelForCausalLM.from_pretrained(<this folder>) and use the chat template.\n"
            "Run python chat_app.py for a chat window.\n"
            "For health education only. Not medical advice.\n")
print(os.listdir(MERGED_DIR))

In [ ]:
from IPython.display import FileLink, display
os.chdir("/kaggle/working")
# weights barely compress, so store without compression (much faster than shutil.make_archive)
with zipfile.ZipFile("menst-slm-merged.zip", "w", zipfile.ZIP_STORED, allowZip64=True) as z:
    for root, _, files in os.walk(MERGED_DIR):
        for fn in files:
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, MERGED_DIR))
print("zip size (GB): %.2f" % (os.path.getsize("menst-slm-merged.zip") / 1e9))
display(FileLink("menst-slm-merged.zip"))

**Downloading**
- Small file (adapter only): `menst-slm-lora.zip`. Needs the base model `MODEL_ID` too.
- Full model: `menst-slm-merged.zip`. Self-contained. It is large (a few GB), so use the Output tab in the right sidebar or commit the notebook with Save Version and download from the Output section. The link above can time out on big files.
- Most reliable for sharing and for deploying a Space: push to the Hugging Face Hub (next cell).

In [ ]:
# Optional: push to the Hugging Face Hub.
# Add a write token as a Kaggle secret named HF_TOKEN (Add-ons -> Secrets), then set PUSH = True.
PUSH = False
HUB_REPO = "your-username/menst-slm"   # change this

if PUSH:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    merged.push_to_hub(HUB_REPO)
    tok.push_to_hub(HUB_REPO)
    print("Pushed to https://huggingface.co/" + HUB_REPO)

## 9. Convert to GGUF (4-bit) for a tiny CPU host

Builds `llama-quantize` from llama.cpp, converts the merged model, and writes two files:
`Q4_K_M` (better quality) and `Q3_K_M` (smaller, use it if the host runs out of memory).
Needs Internet = On (GitHub). Takes a few minutes.

In [ ]:
import subprocess

LCPP = "/kaggle/working/llama.cpp"

def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-2500:])
    if r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")

if not os.path.isdir(LCPP):
    sh(f"git clone --depth 1 https://github.com/ggml-org/llama.cpp {LCPP}")
sh("pip install -q gguf sentencepiece")          # not the repo's requirements file: it would replace Kaggle's torch

F16 = "/kaggle/working/luna-f16.gguf"
sh(f"cd {LCPP} && PYTHONPATH={LCPP}/gguf-py python convert_hf_to_gguf.py {MERGED_DIR} --outfile {F16} --outtype f16")
sh(f"cmake -S {LCPP} -B {LCPP}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release")
sh(f"cmake --build {LCPP}/build --target llama-quantize -j 4")

GGUF_FILES = {}
for q in ("Q4_K_M", "Q3_K_M"):
    outp = f"/kaggle/working/luna-0.5b-{q}.gguf"
    sh(f"{LCPP}/build/bin/llama-quantize {F16} {outp} {q}")
    GGUF_FILES[q] = outp
    print(q, "%.0f MB" % (os.path.getsize(outp) / 1e6))
os.remove(F16)
shutil.rmtree(LCPP, ignore_errors=True)            # keep /kaggle/working small

In [ ]:
# Upload the GGUF files to the Hugging Face Hub so the host can download them.
# (Needs the HF_TOKEN Kaggle secret, same as the earlier push cell.)
PUSH_GGUF = False
GGUF_REPO = "your-username/luna-loves-you-gguf"   # change this

if PUSH_GGUF:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(GGUF_REPO, exist_ok=True)
    for f in GGUF_FILES.values():
        api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=GGUF_REPO)
    print("Uploaded to https://huggingface.co/" + GGUF_REPO)
else:
    print("Set PUSH_GGUF = True to upload, or download the .gguf files from the Output tab.")